# Let's work on finding the correlation between measles coverage and incidence

### Let's use
- rada_notebooks/nis_measles_vacc_coverage.csv (Primary coverage)
- raw/Vaccination_Coverage_and_Exemptions_among_Kindergartners_20260827.csv (Secondary coverage)
- app/data/tycho_measles_control.csv, Tycho Measles (Primary incidents)

### Start with measles incidence/cases

In [5]:
import pandas as pd

tycho_incidents_df = pd.read_csv('../app/data/tycho_measles_control.csv')
tycho_incidents_df.head()

,week,state,cases,year
0,1,NY,376.0,1931
1,1,OR,67.0,1931
2,1,CO,41.0,1931
3,1,AZ,50.0,1931
4,1,MO,1160.0,1931


### Our primary coverage will be NIS Child

In [11]:
nis_coverage_df = pd.read_csv('../app/data/nis_measles_vacc_coverage.csv')
nis_coverage_df.head()

,state,year,n,coverage_pct
0,AK,2015,295.0,89.70
1,AK,2016,288.0,85.83
2,AK,2017,251.0,89.56
3,AK,2018,228.0,85.11
4,AK,2019,240.0,85.79


## Recall what these columns mean:

| Column         | Level          | What it means                              | Why you need it                                                                     |
| -------------- | -------------- | ------------------------------------------ | ----------------------------------------------------------------------------------- |
| **`SEQNUMC`**  | Child          | Unique child identifier                    | Identifies the individual child/record                                              |
| **`SEQNUMHH`** | Household      | Household identifier                       | Identifies which household the child belongs to; used in survey design              |
| **`STRATUM`**  | Sampling group | Survey sampling stratum                    | Identifies the sampling group; needed for correct SEs/CIs                           |
| **`PROVWT`**   | Child          | Provider-phase survey weight               | Determines how much the child contributes to population estimates                   |
| **`P_UTDMCV`** | Child          | Measles vaccination status                 | `1` = received ≥1 qualifying measles-containing vaccination; `0` = did not          |
| **`P_NUMMMR`** | Child          | Number of measles-containing vaccine doses | Shows how many provider-reported measles-containing vaccinations the child received |
| **`STATE`**    | Geography      | State code                                 | Lets you calculate vaccination coverage by state                                    |
| **`YEAR`**     | Time           | Survey year                                | Lets you calculate and compare coverage over time                                   |

In [7]:
assert False

AssertionError: 

In [ ]:
import pandas as pd

schoolvaxview_df = pd.read_csv('../raw/school_vax_view/schoolvaxview.csv')
schoolvaxview_df.head()

,Vaccine/Exemption,Dose,Geography Type,Geography,School Year,Estimate (%),Population Size,Percent Surveyed,Footnotes,Number of Exemptions,Survey Type
0,MMR,NaN,States,Kansas,2022-23,91.6,35543.0,30.8,†. ‡. §,NaN,Stratified 1-stage cluster sample
1,MMR,NaN,States,Kentucky,2022-23,90.1,54742.0,96.9,>=. ‡. §,NaN,Census
2,MMR,NaN,States,Louisiana,2022-23,92.2,54314.0,100.0,*,NaN,Census
3,MMR,NaN,States,Maine,2022-23,96.8,12403.0,93.9,NaN,NaN,Census
4,MMR,NaN,States,Maryland,2022-23,96.7,59684.0,100.0,*. ‡,NaN,Census


In [ ]:
schoolvaxview_df["year"] = (
    schoolvaxview_df["School Year"]
    .str.split("-")
    .str[0]
    .astype(int)
    .add(1)
)

In [ ]:
import us


schoolvaxview_df["Geography"] = schoolvaxview_df["Geography"].map(
    lambda x: us.states.lookup(x).abbr if us.states.lookup(x) else None
)


schoolvaxview_df = schoolvaxview_df.rename(columns={'Geography': 'state'})

In [ ]:
core_columns = [
    'Vaccine/Exemption',
    'Dose',
    'Geography Type',
    'School Year',
    'Estimate (%)',
    'year',
    'state',
]

schoolvaxview_df = schoolvaxview_df[core_columns]
schoolvaxview_df.head()

,Vaccine/Exemption,Dose,Geography Type,School Year,Estimate (%),year,state
0,MMR,NaN,States,2022-23,91.6,2023,KS
1,MMR,NaN,States,2022-23,90.1,2023,KY
2,MMR,NaN,States,2022-23,92.2,2023,LA
3,MMR,NaN,States,2022-23,96.8,2023,ME
4,MMR,NaN,States,2022-23,96.7,2023,MD


In [ ]:
measles_schoolvaxview_df = schoolvaxview_df[(schoolvaxview_df['Vaccine/Exemption'] == 'MMR')
                                            & (schoolvaxview_df['Geography Type'] == 'States')]
len(measles_schoolvaxview_df)

868

In [ ]:
measles_schoolvaxview_df.head()

,Vaccine/Exemption,Dose,Geography Type,School Year,Estimate (%),year,state
0,MMR,NaN,States,2022-23,91.6,2023,KS
1,MMR,NaN,States,2022-23,90.1,2023,KY
2,MMR,NaN,States,2022-23,92.2,2023,LA
3,MMR,NaN,States,2022-23,96.8,2023,ME
4,MMR,NaN,States,2022-23,96.7,2023,MD
